# 🚀 Generic Ollama with Ngrok (Any Model)

This notebook runs **any Ollama model** with proper warmup to avoid timeouts.

## Supported Models
- **Qwen**: `qwen3:latest`, `qwen3:8b`, `qwen3:7b`
- **Llama**: `llama4:latest`, `llama3:8b`, `llama3:70b`
- **Mistral**: `ministral-3:latest`, `mistral:7b`
- **GPT-OSS**: `gpt-oss:20b`
- **Any other**: Check [Ollama Library](https://ollama.com/library)

## Features
✅ **Configurable model** - Change `MODEL_NAME` below
✅ **Longer timeouts** - 120s for first request
✅ **Model warmup** - Loads model into GPU before testing
✅ **Better error handling**

---

## ⚙️ Configuration

**Change the model name here:**

In [ ]:
# ============================================================
# CONFIGURATION - Change these values
# ============================================================

# Model to use (see https://ollama.com/library for available models)
MODEL_NAME = "qwen3:latest"  # Change this to any model you want

# Examples:
# MODEL_NAME = "llama4:latest"
# MODEL_NAME = "ministral-3:latest"
# MODEL_NAME = "gpt-oss:20b"
# MODEL_NAME = "mistral:7b"
# MODEL_NAME = "qwen3:8b"

# Ngrok configuration (optional - use your own domain)
NGROK_AUTH_TOKEN = "2zx1rwuzaVjFIfTotSfdqXHOPSR_ofAcTHk35E86USHnRxmo"
STATIC_DOMAIN = "allegedly-hopeful-stallion.ngrok-free.app"

# Timeout settings
WARMUP_TIMEOUT = 120  # seconds for first request
NORMAL_TIMEOUT = 60   # seconds for regular requests

print(f"✅ Configuration set")
print(f"📦 Model: {MODEL_NAME}")
print(f"🌐 Domain: {STATIC_DOMAIN}")

## Step 1: Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"\nGPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB" if torch.cuda.is_available() else "N/A")

## Step 2: Install

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q pyngrok
print("✓ Installed")

## Step 3: Configure Ngrok

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print(f"✓ Ngrok: {STATIC_DOMAIN}")

## Step 4: Start Ollama

In [ ]:
import time

print("🚀 Starting Ollama...")
get_ipython().system_raw('OLLAMA_HOST=0.0.0.0:11434 ollama serve > /tmp/ollama.log 2>&1 &')
time.sleep(10)

!ps aux | grep ollama | grep -v grep
print("\n✓ Ollama running")

## Step 5: Pull Model

In [ ]:
print(f"📥 Pulling {MODEL_NAME}...\n")
!ollama pull {MODEL_NAME}
print(f"\n✓ Model {MODEL_NAME} downloaded")

## Step 6: Warmup Model (IMPORTANT!)

**This loads the model into GPU memory to avoid timeouts later.**

In [ ]:
import requests

print(f"🔥 Warming up {MODEL_NAME} (this takes ~30-60 seconds)...")
print("This is normal - first inference loads model into GPU\n")

warmup_url = "http://localhost:11434/api/generate"
warmup_payload = {
    "model": MODEL_NAME,
    "prompt": "Hi",
    "stream": False
}

try:
    response = requests.post(warmup_url, json=warmup_payload, timeout=WARMUP_TIMEOUT)
    if response.status_code == 200:
        print("✓ Model warmed up and ready!")
    else:
        print(f"⚠ Warmup returned status {response.status_code}")
except Exception as e:
    print(f"⚠ Warmup timeout (this is OK, model might still work): {e}")

print("\nWaiting 5 more seconds...")
time.sleep(5)
print("✓ Ready for testing")

## Step 7: Create Tunnel

In [ ]:
from IPython.display import display, HTML

OLLAMA_PORT = 11434
print("🌐 Creating ngrok tunnel...")

public_url = ngrok.connect(OLLAMA_PORT, domain=STATIC_DOMAIN)
tunnel_url = f"https://{STATIC_DOMAIN}"

print("\n" + "="*70)
print("✓ READY!")
print("="*70)
print(f"\nAPI Endpoint: {tunnel_url}/v1")
print(f"Model: {MODEL_NAME}")
print(f"\nAdd to .env:")
print(f"  BASE_URL={tunnel_url}/v1")
print(f"  API_KEY=ollama")
print(f"  MODEL_NAME={MODEL_NAME}")
print("="*70)

display(HTML(f'''
<div style="background:#e7f3ff;padding:20px;border-radius:10px;border:2px solid #0066cc">
    <h3 style="color:#0066cc;margin-top:0">🎉 {MODEL_NAME} Ready!</h3>
    <p><strong>Endpoint:</strong> <code>{tunnel_url}/v1</code></p>
    <p><strong>Model:</strong> <code>{MODEL_NAME}</code></p>
</div>
'''))

## Step 8: Test Chat

In [ ]:
import json

def test_chat(prompt, timeout=None):
    if timeout is None:
        timeout = NORMAL_TIMEOUT
    
    url = f"http://localhost:{OLLAMA_PORT}/v1/chat/completions"
    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 200
    }

    try:
        response = requests.post(url, json=payload, timeout=timeout)
        response.raise_for_status()
        result = response.json()
        return result['choices'][0]['message']['content']
    except requests.exceptions.Timeout:
        return f"ERROR: Timeout - model might be too large for T4 GPU (tried {timeout}s)"
    except Exception as e:
        return f"ERROR: {str(e)}"

print(f"Testing {MODEL_NAME} chat...\n")
response = test_chat("What is 2+2? Answer briefly.")
print(f"Response: {response}\n")

if "ERROR" not in response:
    print("✅ Chat works!")
else:
    print("❌ Chat failed - see troubleshooting below")

## Step 9: Test Tool Calling

In [ ]:
def test_tools():
    url = f"http://localhost:{OLLAMA_PORT}/v1/chat/completions"
    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": "What's the weather in Tokyo?"}],
        "tools": [{
            "type": "function",
            "function": {
                "name": "get_weather",
                "description": "Get weather",
                "parameters": {
                    "type": "object",
                    "properties": {"location": {"type": "string"}},
                    "required": ["location"]
                }
            }
        }],
        "tool_choice": "auto"
    }

    try:
        response = requests.post(url, json=payload, timeout=NORMAL_TIMEOUT)
        response.raise_for_status()
        result = response.json()

        print("Response:")
        print(json.dumps(result, indent=2))

        if 'tool_calls' in result.get('choices', [{}])[0].get('message', {}):
            print("\n" + "="*70)
            print("✅ TOOL CALLING WORKS!")
            print("="*70)
            print("🎉 Can be used for browser automation!")
            return True
        else:
            print("\n" + "="*70)
            print("❌ NO TOOL CALLING")
            print("="*70)
            print(f"⚠️ {MODEL_NAME} might not support tool calling")
            print("Note: Not all models support tool calling. Check model documentation.")
            return False
    except Exception as e:
        print(f"\n❌ Tool test failed: {e}")
        return False

print(f"🧪 Testing tool calling for {MODEL_NAME}...\n")
test_tools()

## Troubleshooting

### Still Getting Timeouts?
1. **T4 GPU might be too small** - Try a smaller model variant
   - Instead of `llama4:latest`, try `llama3:8b`
   - Instead of `qwen3:latest`, try `qwen3:8b` or `qwen3:7b`
   - Instead of `mistral:latest`, try `mistral:7b`

2. **Check available models**: Visit [Ollama Library](https://ollama.com/library)

3. **Model doesn't exist**: Make sure the model name is correct
   - Run `!ollama list` to see available models
   - Check spelling and version tag

4. **Tool calling not supported**: Not all models support tool calling
   - Qwen models: ✅ Support tool calling
   - Llama models: ✅ Support tool calling
   - Some smaller models: ❌ May not support it

### Popular Model Recommendations
**For T4 GPU (15GB VRAM):**
- `qwen3:8b` - Fast, supports tool calling
- `llama3:8b` - Reliable, supports tool calling
- `mistral:7b` - Fast, good quality
- `ministral-3:latest` - Small and efficient

**For A100 GPU (40GB VRAM):**
- `llama4:latest` - Best quality
- `qwen3:latest` - Fast and accurate
- `gpt-oss:20b` - Large context window

## Monitor & Info

In [ ]:
print("Available models:")
!ollama list
print(f"\nCurrent model: {MODEL_NAME}")
print(f"Tunnel: {tunnel_url}/v1")
print(f"\nAPI Configuration:")
print(f"  BASE_URL={tunnel_url}/v1")
print(f"  API_KEY=ollama")
print(f"  MODEL_NAME={MODEL_NAME}")

## 📊 Real-Time Request/Response Logger

Monitor all incoming requests and responses in real-time!

In [ ]:
# Install required packages for logging
!pip install -q flask flask-cors

from flask import Flask, request, Response
from flask_cors import CORS
import requests as req
import json
from datetime import datetime
import threading

# Create Flask app for logging proxy
app = Flask(__name__)
CORS(app)

# Store logs in memory
request_logs = []
MAX_LOGS = 100  # Keep last 100 requests

def log_request(method, path, headers, body, response_data, status_code, duration):
    """Log request and response details"""
    log_entry = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3],
        "method": method,
        "path": path,
        "headers": dict(headers),
        "request_body": body,
        "response_body": response_data,
        "status_code": status_code,
        "duration_ms": round(duration * 1000, 2)
    }
    
    request_logs.append(log_entry)
    if len(request_logs) > MAX_LOGS:
        request_logs.pop(0)
    
    # Print to console
    print("\n" + "="*80)
    print(f"🔵 [{log_entry['timestamp']}] {method} {path}")
    print("="*80)
    print(f"\n📤 REQUEST:")
    print(json.dumps(body, indent=2))
    print(f"\n📥 RESPONSE ({status_code}) - {log_entry['duration_ms']}ms:")
    print(json.dumps(response_data, indent=2))
    print("="*80 + "\n")

@app.route('/logs', methods=['GET'])
def get_logs():
    """Endpoint to retrieve logs"""
    return json.dumps(request_logs, indent=2)

@app.route('/logs/clear', methods=['POST'])
def clear_logs():
    """Clear all logs"""
    request_logs.clear()
    return {"status": "cleared"}

@app.route('/<path:path>', methods=['GET', 'POST', 'PUT', 'DELETE', 'PATCH'])
def proxy(path):
    """Proxy all requests to Ollama and log them"""
    import time
    start_time = time.time()
    
    # Forward request to Ollama
    ollama_url = f"http://localhost:11434/{path}"
    
    # Get request data
    request_body = None
    if request.data:
        try:
            request_body = json.loads(request.data)
        except:
            request_body = request.data.decode('utf-8')
    
    # Forward request
    try:
        response = req.request(
            method=request.method,
            url=ollama_url,
            headers={k: v for k, v in request.headers if k.lower() != 'host'},
            data=request.data,
            params=request.args,
            stream=True
        )
        
        # Collect response
        response_data = response.content
        try:
            response_json = json.loads(response_data)
        except:
            response_json = response_data.decode('utf-8')
        
        duration = time.time() - start_time
        
        # Log it
        log_request(
            method=request.method,
            path=path,
            headers=request.headers,
            body=request_body,
            response_data=response_json,
            status_code=response.status_code,
            duration=duration
        )
        
        # Return response
        return Response(
            response_data,
            status=response.status_code,
            headers=dict(response.headers)
        )
    except Exception as e:
        print(f"❌ Proxy error: {e}")
        return {"error": str(e)}, 500

def run_logger():
    """Run Flask app in background"""
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

# Start logger in background thread
logger_thread = threading.Thread(target=run_logger, daemon=True)
logger_thread.start()

print("✅ Real-time logger started on port 5000")
print("\n📊 Logs will appear below each request")
print("\nEndpoints:")
print("  - GET  /logs       -> View all logs")
print("  - POST /logs/clear -> Clear logs")

## 🌐 Expose Logger with Ngrok

In [ ]:
import time
time.sleep(3)  # Wait for Flask to start

# Create tunnel for logger
logger_tunnel = ngrok.connect(5000)
logger_url = logger_tunnel.public_url

print("\n" + "="*70)
print("📊 LOGGER ENDPOINTS")
print("="*70)
print(f"\n🔍 View logs: {logger_url}/logs")
print(f"🗑️  Clear logs: {logger_url}/logs/clear (POST)")
print(f"\n⚠️  IMPORTANT: Use the logger proxy for requests:")
print(f"   OLD: {tunnel_url}/v1")
print(f"   NEW: {logger_url}/v1  <- Use this to see logs")
print("\n" + "="*70)

display(HTML(f'''
<div style="background:#fff3cd;padding:20px;border-radius:10px;border:2px solid #ffc107;margin-top:20px">
    <h3 style="color:#856404;margin-top:0">📊 Real-Time Logging Active!</h3>
    <p><strong>View Logs:</strong> <a href="{logger_url}/logs" target="_blank">{logger_url}/logs</a></p>
    <p><strong>Use This Endpoint:</strong> <code>{logger_url}/v1</code></p>
    <p style="margin-bottom:0"><em>All requests and responses will be logged in real-time below!</em></p>
</div>
'''))

## 🧪 Test Logger

Make a test request to see the logging in action!

In [ ]:
import requests

print("🧪 Sending test request through logger...\n")

# Use the logger URL instead of direct Ollama
test_url = f"{logger_url}/v1/chat/completions"

test_payload = {
    "model": MODEL_NAME,
    "messages": [{"role": "user", "content": "Say hello!"}],
    "max_tokens": 50
}

response = requests.post(test_url, json=test_payload, timeout=60)
print(f"\n✅ Response received!")
print(f"\n📋 Check the logs above to see the full request/response details")
print(f"\n🌐 Or visit: {logger_url}/logs")

## 📋 View All Logs

In [ ]:
# Fetch and display all logs
logs_response = requests.get(f"{logger_url}/logs")
logs = logs_response.json()

print(f"📊 Total requests logged: {len(logs)}\n")

if logs:
    print("Last 5 requests:\n")
    for log in logs[-5:]:
        print(f"[{log['timestamp']}] {log['method']} {log['path']} - {log['status_code']} ({log['duration_ms']}ms)")
else:
    print("No requests logged yet. Make a request to see logs!")

## 🗑️ Clear Logs

In [ ]:
# Clear all logs
response = requests.post(f"{logger_url}/logs/clear")
print("✅ Logs cleared!")

## Quick Model Change

Want to try a different model? Run this cell:

In [ ]:
# Change model without restarting everything
NEW_MODEL = "llama3:8b"  # Change this to any model

print(f"📥 Pulling {NEW_MODEL}...")
!ollama pull {NEW_MODEL}

MODEL_NAME = NEW_MODEL
print(f"\n✅ Model changed to: {MODEL_NAME}")
print(f"🌐 Same endpoint: {tunnel_url}/v1")
print(f"\nUpdate your .env:")
print(f"  MODEL_NAME={MODEL_NAME}")